#  대화 이력 관리를 위한 메모리 구현(Chat History)

### **학습 목표:**  대화 이력 관리를 위한 메모리 컴포넌트 구현 방법을 실습한다

---

# 환경 설정 및 준비

`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
from glob import glob

from pprint import pprint
import json

`(3) LLM 설정`

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='gpt-4.1-mini',
    temperature=0.3,
    top_p=0.9, 
)

# **채팅 히스토리 관리**

* 채팅봇의 핵심 기능은 이전 대화 내용을 문맥으로 활용하는 것으로, 이를 통해 자연스러운 대화 흐름을 유지할 수 있습니다.

* 가장 기본적인 방식은 이전 메시지들을 모델의 프롬프트에 직접 포함시키는 것이지만, 오래된 메시지는 적절히 제거하여 모델이 처리해야 할 정보량을 조절할 수 있습니다.

* 장시간 진행되는 대화의 경우, 단순히 이전 메시지를 저장하는 것을 넘어 대화 내용을 요약하여 저장하는 등의 고급 메모리 관리 기법을 활용할 수 있습니다.

### 1. **메시지 전달 방식(Message Passing)**

* 메시지 전달 방식은 LangChain에서 가장 기본적인 메모리 구현 방법으로, 이전 대화 기록(chat history)을 체인에 직접 전달하여 문맥을 유지하는 방식입니다.

* 이 방식은 SystemMessage(시스템 지시사항), HumanMessage(사용자 입력), AIMessage(AI 응답) 등 다양한 유형의 메시지를 ChatPromptTemplate을 통해 구조화하며, MessagesPlaceholder를 사용하여 이전 대화 내용을 포함시킵니다.

* 챗봇의 기본적인 메모리 시스템을 구현하는데 사용되며, 이를 통해 AI는 이전 대화 맥락을 이해하고 그에 맞는 적절한 응답을 생성할 수 있습니다.


In [4]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# ChatPromptTemplate를 사용하여 챗봇의 초기 메시지를 정의
prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content="You are a helpful assistant."),
    MessagesPlaceholder(variable_name="messages"),    # 메시지 목록을 동적으로 삽입하는 부분
])

# ChatPromptTemplate에 삽입할 메시지 목록을 정의
messages = [
        HumanMessage(content="안녕하세요. 제 이름은 홍길동입니다."),
        AIMessage(content="안녕하세요! 어떻게 도와드릴까요?"),
]    

# ChatPromptTemplate에 삽입할 메시지 목록을 업데이트하고 출력 
pprint(prompt.format(messages=messages))

('System: You are a helpful assistant.\n'
 'Human: 안녕하세요. 제 이름은 홍길동입니다.\n'
 'AI: 안녕하세요! 어떻게 도와드릴까요?')


In [5]:
# 대화형 체인을 정의 (prompt -> llm)
chain = prompt | llm

# 기본적인 메시지 전달: 이전 메시지 목록에 새로운 메시지를 추가해서 전달
response = chain.invoke({
    "messages": messages + [HumanMessage(content="제 이름을 기억하나요?")] # 이전 메시지를 기억하는지 확인하는 질문 메시지 추가
})

pprint(response)

AIMessage(content='네, 제 기억에 따르면 당신의 이름은 홍길동입니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 53, 'total_tokens': 75, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_6d7dcc9a98', 'id': 'chatcmpl-CEvVrAFvqL8s82ETPc1x4ZF0NUhvf', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--903e9f87-b302-4d69-b102-e7d8a3993923-0', usage_metadata={'input_tokens': 53, 'output_tokens': 22, 'total_tokens': 75, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})


### 2. **RunnableWithMessageHistory**

* RunnableWithMessageHistory는 LangChain에서 대화 기록을 관리하는 고급 기능으로, 체인의 실행 과정에서 메시지 기록을 자동으로 저장하고 검색할 수 있게 해주는 래퍼(wrapper) 클래스입니다.

* 이 기능은 대화 세션별로 독립적인 기록을 유지할 수 있게 해주며, ConfigurableField를 통해 메모리 구성을 유연하게 조정할 수 있습니다. 특히 여러 사용자와 동시에 대화할 때 각 세션의 컨텍스트를 분리하여 관리하는 데 매우 유용합니다.

* 주요 장점은 대화 기록 관리의 자동화와 일관성 있는 메시지 처리이지만, 메모리 저장소 설정과 관리에 추가적인 구성이 필요하다는 점을 고려해야 합니다. 또한 Redis나 다른 외부 저장소와 통합하여 영구적인 대화 기록 보관도 가능합니다.

* 구현 시에는 get_session_history 콜백을 통해 세션ID별로 메시지 기록을 관리하며, 이를 통해 각 대화의 컨텍스트를 정확하게 유지할 수 있습니다.


`(1) 메모리 기반 로컬 저장소 활용`

* `InMemoryHistory` 클래스는 대화 이력의 기본 구조를 제공하며, BaseChatMessageHistory와 BaseModel을 상속받아 메시지를 메모리에서 효율적으로 관리합니다. 특히 messages 리스트를 통해 BaseMessage 객체들을 순차적으로 저장하고, add_messages와 clear 메서드로 히스토리를 유연하게 관리할 수 있습니다.

* 시스템의 핵심인 `store` 변수는 전역 딕셔너리로 구현되어 세션별 대화 이력을 구분하여 저장합니다. session_id를 키로 사용하여 각 세션의 InMemoryHistory 객체에 빠르게 접근할 수 있으며, 이를 통해 다중 사용자 환경에서도 효율적인 대화 관리가 가능합니다.

* `get_session_history` 함수는 세션 관리의 진입점 역할을 하며, 존재하지 않는 세션에 대해 자동으로 새로운 InMemoryHistory 객체를 생성하는 팩토리 패턴을 구현합니다. 이러한 구조를 통해 세션의 생명주기를 자동으로 관리하고 메모리 효율성을 높일 수 있습니다.

In [6]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage
from pydantic import BaseModel, Field
from typing import List

# 메모리 기반 히스토리 구현
class InMemoryHistory(BaseChatMessageHistory, BaseModel):
    messages: List[BaseMessage] = Field(default_factory=list)
    
    def add_messages(self, messages: List[BaseMessage]) -> None:
        self.messages.extend(messages)
    
    def clear(self) -> None:
        self.messages = []

# 세션 저장소
store = {}

# 세션 ID로 히스토리 가져오기
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryHistory()
    return store[session_id]

In [7]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

# 프롬프트 템플릿 설정
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 여행 가이드입니다. 관광객에게 유용한 정보를 제공하세요."),
    MessagesPlaceholder(variable_name="history"), 
    ("human", "{input}")
])

# 프롬프트와 llm을 연결하여 체인 생성
chain = prompt | llm

# 히스토리 관리 추가  
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# 지정된 세션 ID(tourist_1)를 사용하여체인 실행
response = chain_with_history.invoke(
    {
        "input": "서울에서 가볼만한 곳을 추천해주세요."
    },
    config={"configurable": {"session_id": "tourist_1"}}
)


print(f"여행 가이드 답변:\n{response.content}")

여행 가이드 답변:
서울에서 가볼 만한 곳을 추천해드릴게요!

1. 경복궁 – 조선 시대의 대표 궁궐로, 전통 건축과 아름다운 정원을 감상할 수 있어요.
2. 북촌 한옥마을 – 전통 한옥이 잘 보존된 마을로, 한국의 옛 생활문화를 체험하기 좋아요.
3. 명동 – 쇼핑과 맛집이 가득한 번화가로, 다양한 패션 아이템과 길거리 음식을 즐길 수 있어요.
4. N서울타워 – 서울 전경을 한눈에 볼 수 있는 전망대로, 특히 야경이 아름답습니다.
5. 홍대 – 젊음의 거리로 유명하며, 예술과 음악, 카페 문화가 활발한 곳이에요.
6. 동대문 디자인 플라자(DDP) – 현대적인 건축물과 다양한 전시, 패션 마켓이 열리는 공간입니다.
7. 한강공원 – 강변을 따라 산책하거나 자전거 타기, 피크닉을 즐기기에 좋은 장소입니다.

필요하시면 각 장소별 교통편이나 추천 일정도 알려드릴게요!


In [8]:
# 대화 히스토리 출력 
history = get_session_history("tourist_1")

pprint(history.messages)

[HumanMessage(content='서울에서 가볼만한 곳을 추천해주세요.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='서울에서 가볼 만한 곳을 추천해드릴게요!\n\n1. 경복궁 – 조선 시대의 대표 궁궐로, 전통 건축과 아름다운 정원을 감상할 수 있어요.\n2. 북촌 한옥마을 – 전통 한옥이 잘 보존된 마을로, 한국의 옛 생활문화를 체험하기 좋아요.\n3. 명동 – 쇼핑과 맛집이 가득한 번화가로, 다양한 패션 아이템과 길거리 음식을 즐길 수 있어요.\n4. N서울타워 – 서울 전경을 한눈에 볼 수 있는 전망대로, 특히 야경이 아름답습니다.\n5. 홍대 – 젊음의 거리로 유명하며, 예술과 음악, 카페 문화가 활발한 곳이에요.\n6. 동대문 디자인 플라자(DDP) – 현대적인 건축물과 다양한 전시, 패션 마켓이 열리는 공간입니다.\n7. 한강공원 – 강변을 따라 산책하거나 자전거 타기, 피크닉을 즐기기에 좋은 장소입니다.\n\n필요하시면 각 장소별 교통편이나 추천 일정도 알려드릴게요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 261, 'prompt_tokens': 40, 'total_tokens': 301, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_6d7dcc9a98', 'id': 'chatcmpl-CEvVzB2X4Ni

In [9]:
# 이전 대화 내용을 기반으로 새로운 질문을 추가하여 체인 실행
response = chain_with_history.invoke(
    {
        "input": "이전에 추천한 장소 중에서 가장 인기 있는 곳은 어디인가요?"
    },
    config={"configurable": {"session_id": "tourist_1"}}
)

print(f"여행 가이드 답변:\n{response.content}")

여행 가이드 답변:
이전에 추천드린 장소 중에서 가장 인기 있는 곳은 경복궁과 N서울타워입니다.

- **경복궁**은 한국의 대표적인 역사 유적지로, 많은 관광객들이 한국 전통문화를 체험하고 사진을 찍기 위해 방문합니다.
- **N서울타워**는 서울의 랜드마크 중 하나로, 서울 전경과 야경을 감상할 수 있어 특히 저녁 시간대에 매우 인기가 많습니다.

두 곳 모두 서울을 처음 방문하는 관광객들에게 꼭 추천되는 명소입니다. 방문 시간과 관심사에 따라 선택하시면 좋을 것 같아요!


In [10]:
# 대화 히스토리 출력
history = get_session_history("tourist_1")

pprint(history.messages)

[HumanMessage(content='서울에서 가볼만한 곳을 추천해주세요.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='서울에서 가볼 만한 곳을 추천해드릴게요!\n\n1. 경복궁 – 조선 시대의 대표 궁궐로, 전통 건축과 아름다운 정원을 감상할 수 있어요.\n2. 북촌 한옥마을 – 전통 한옥이 잘 보존된 마을로, 한국의 옛 생활문화를 체험하기 좋아요.\n3. 명동 – 쇼핑과 맛집이 가득한 번화가로, 다양한 패션 아이템과 길거리 음식을 즐길 수 있어요.\n4. N서울타워 – 서울 전경을 한눈에 볼 수 있는 전망대로, 특히 야경이 아름답습니다.\n5. 홍대 – 젊음의 거리로 유명하며, 예술과 음악, 카페 문화가 활발한 곳이에요.\n6. 동대문 디자인 플라자(DDP) – 현대적인 건축물과 다양한 전시, 패션 마켓이 열리는 공간입니다.\n7. 한강공원 – 강변을 따라 산책하거나 자전거 타기, 피크닉을 즐기기에 좋은 장소입니다.\n\n필요하시면 각 장소별 교통편이나 추천 일정도 알려드릴게요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 261, 'prompt_tokens': 40, 'total_tokens': 301, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_6d7dcc9a98', 'id': 'chatcmpl-CEvVzB2X4Ni

`(2) SQLite 데이터베이스 영구 저장소 활용`

* SQLite 통합 구현을 위해서는 먼저 메시지를 저장할 데이터베이스 테이블 구조를 정의하고, BaseChatMessageHistory를 상속받아 메시지 저장/조회 로직을 구현해야 합니다. 이때 세션 ID를 기준으로 대화 내용을 구분하여 관리합니다.

* 시스템 구현 시에는 메시지의 타입(Human/AI), 내용, 메타데이터, 타임스탬프 등의 정보를 체계적으로 저장하고, 필요할 때 효율적으로 검색하고 활용할 수 있도록 적절한 인덱싱 전략을 수립하는 것이 중요합니다.

In [11]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
import sqlite3
from typing import List
import json

class SQLiteChatMessageHistory(BaseChatMessageHistory):
    """ 
    SQLite 데이터베이스를 사용하여 챗봇 대화 히스토리를 저장하는 클래스

    Attributes:
        session_id (str): 세션 ID
        db_path (str): SQLite 데이터베이스 파일 경로

    """
    def __init__(self, session_id: str, db_path: str = "chat_history.db"):
        self.session_id = session_id
        self.db_path = db_path
        self._create_tables()
    
    def _create_tables(self):
        """데이터베이스 테이블 생성"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        # 메시지 테이블 생성
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS messages (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                session_id TEXT,
                message_type TEXT,
                content TEXT,
                metadata TEXT,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """)
        
        conn.commit()
        conn.close()
    
    def add_message(self, message: BaseMessage) -> None:
        """단일 메시지 추가"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        cursor.execute("""
            INSERT INTO messages (session_id, message_type, content, metadata)
            VALUES (?, ?, ?, ?)
        """, (
            self.session_id,
            message.__class__.__name__,
            message.content,
            json.dumps(message.additional_kwargs)
        ))
        
        conn.commit()
        conn.close()
    
    def add_messages(self, messages: List[BaseMessage]) -> None:
        """여러 메시지 추가"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        for message in messages:
            cursor.execute("""
                INSERT INTO messages (session_id, message_type, content, metadata)
                VALUES (?, ?, ?, ?)
            """, (
                self.session_id,
                message.__class__.__name__,
                message.content,
                json.dumps(message.additional_kwargs)
            ))
        
        conn.commit()
        conn.close()
    
    def clear(self) -> None:
        """세션의 모든 메시지 삭제"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        cursor.execute("""
            DELETE FROM messages WHERE session_id = ?
        """, (self.session_id,))
        
        conn.commit()
        conn.close()
    
    @property
    def messages(self) -> List[BaseMessage]:
        """저장된 메시지 조회"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        cursor.execute("""
            SELECT message_type, content, metadata
            FROM messages 
            WHERE session_id = ?
            ORDER BY created_at
        """, (self.session_id,))
        
        messages = []
        for row in cursor.fetchall():
            message_type, content, metadata = row
            if message_type == "HumanMessage":
                message = HumanMessage(content=content)
            else:
                message = AIMessage(content=content)
            
            if metadata:
                message.additional_kwargs = json.loads(metadata)
            
            messages.append(message)
        
        conn.close()
        return messages
    

# 세션 ID로 히스토리 가져오기
def get_chat_history(session_id: str) -> BaseChatMessageHistory:
    return SQLiteChatMessageHistory(session_id=session_id)

In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

# 프롬프트 템플릿 설정
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 여행 가이드입니다. 관광객에게 유용한 정보를 제공하세요."),
    MessagesPlaceholder(variable_name="history"), 
    ("human", "{input}")
])

# 프롬프트와 llm을 연결하여 체인 생성
chain = prompt | llm

# 히스토리 관리 추가  
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_chat_history,
    input_messages_key="input",
    history_messages_key="history"
)

# 지정된 세션 ID(tourist_1)를 사용하여체인 실행
response = chain_with_history.invoke(
    {
        "input": "수원에서 가볼만한 곳을 추천해주세요."
    },
    config={"configurable": {"session_id": "tourist_1"}}
)


print(f"여행 가이드 답변:\n{response.content}")

여행 가이드 답변:
수원은 역사와 문화가 풍부한 도시로, 다양한 관광 명소가 있습니다. 수원에서 가볼 만한 곳 몇 곳을 추천해드릴게요.

1. 수원 화성  
- 조선 시대 정조대왕이 세운 성곽으로, 유네스코 세계문화유산에 등재되어 있습니다. 성곽을 따라 걷거나 화성행궁, 장안문, 화서문 등 주요 건축물을 둘러보세요.  
- 특히 야경이 아름다워 저녁 시간 방문도 추천합니다.

2. 화성행궁  
- 조선시대 임시 궁궐로 사용된 곳으로, 전통 건축물과 함께 다양한 문화 체험 프로그램이 열립니다.  
- 한복 대여 후 사진 찍기 좋은 장소입니다.

3. 수원 박물관  
- 수원의 역사와 문화를 한눈에 볼 수 있는 박물관입니다. 가족 단위 방문객에게도 좋고, 다양한 전시와 체험 프로그램이 마련되어 있습니다.

4. 광교호수공원  
- 자연과 함께 산책하기 좋은 공원으로, 호수 주변을 걷거나 자전거를 탈 수 있습니다. 카페와 음식점도 많아 휴식하기 좋습니다.

5. 행궁동 카페거리  
- 아기자기한 카페와 맛집이 모여 있는 곳으로, 젊은 층과 관광객에게 인기 있는 장소입니다.

수원 방문 시 교통편은 지하철 1호선과 분당선을 이용하면 편리하며, 버스도 잘 연결되어 있습니다. 즐거운 여행 되세요!


In [13]:
# 대화 히스토리 출력
history = get_chat_history("tourist_1")

pprint(history.messages)

[HumanMessage(content='수원에서 가볼만한 곳을 추천해주세요.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='수원은 역사와 문화가 풍부한 도시로, 다양한 관광 명소가 있습니다. 수원에서 가볼 만한 곳 몇 곳을 추천해드릴게요.\n\n1. 수원 화성  \n- 조선 시대 정조대왕이 세운 성곽으로, 유네스코 세계문화유산에 등재되어 있습니다. 성곽을 따라 걷거나 화성행궁, 장안문, 화서문 등 주요 건축물을 둘러보세요.  \n- 특히 야경이 아름다워 저녁 시간 방문도 추천합니다.\n\n2. 화성행궁  \n- 조선시대 임시 궁궐로 사용된 곳으로, 전통 건축물과 함께 다양한 문화 체험 프로그램이 열립니다.  \n- 한복 대여 후 사진 찍기 좋은 장소입니다.\n\n3. 수원 박물관  \n- 수원의 역사와 문화를 한눈에 볼 수 있는 박물관입니다. 가족 단위 방문객에게도 좋고, 다양한 전시와 체험 프로그램이 마련되어 있습니다.\n\n4. 광교호수공원  \n- 자연과 함께 산책하기 좋은 공원으로, 호수 주변을 걷거나 자전거를 탈 수 있습니다. 카페와 음식점도 많아 휴식하기 좋습니다.\n\n5. 행궁동 카페거리  \n- 아기자기한 카페와 맛집이 모여 있는 곳으로, 젊은 층과 관광객에게 인기 있는 장소입니다.\n\n수원 방문 시 교통편은 지하철 1호선과 분당선을 이용하면 편리하며, 버스도 잘 연결되어 있습니다. 즐거운 여행 되세요!', additional_kwargs={'refusal': None}, response_metadata={})]


In [14]:
# 이전 대화 내용을 기반으로 새로운 질문을 추가하여 체인 실행
response = chain_with_history.invoke(
    {
        "input": "이전에 추천한 장소 중에서 가장 인기 있는 곳은 어디인가요?"
    },
    config={"configurable": {"session_id": "tourist_1"}}
)

print(f"여행 가이드 답변:\n{response.content}")

여행 가이드 답변:
이전에 추천해드린 장소 중에서 가장 인기 있는 곳은 **수원 화성**입니다.  

수원 화성은 역사적 가치가 뛰어나고, 아름다운 성곽과 다양한 건축물이 잘 보존되어 있어 많은 관광객이 찾는 명소입니다. 특히 성곽을 따라 걷는 산책로와 야경이 매우 아름다워 사진 촬영 장소로도 인기가 높습니다. 또한, 유네스코 세계문화유산으로 지정되어 있어 국내외 관광객 모두에게 사랑받는 곳입니다.  

화성행궁과 함께 방문하면 더욱 풍성한 역사 체험이 가능하니, 수원 여행 시 꼭 들러보시길 추천드립니다!


In [15]:
# 대화 히스토리 출력
history = get_chat_history("tourist_1")

pprint(history.messages)

[HumanMessage(content='수원에서 가볼만한 곳을 추천해주세요.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='수원은 역사와 문화가 풍부한 도시로, 다양한 관광 명소가 있습니다. 수원에서 가볼 만한 곳 몇 곳을 추천해드릴게요.\n\n1. 수원 화성  \n- 조선 시대 정조대왕이 세운 성곽으로, 유네스코 세계문화유산에 등재되어 있습니다. 성곽을 따라 걷거나 화성행궁, 장안문, 화서문 등 주요 건축물을 둘러보세요.  \n- 특히 야경이 아름다워 저녁 시간 방문도 추천합니다.\n\n2. 화성행궁  \n- 조선시대 임시 궁궐로 사용된 곳으로, 전통 건축물과 함께 다양한 문화 체험 프로그램이 열립니다.  \n- 한복 대여 후 사진 찍기 좋은 장소입니다.\n\n3. 수원 박물관  \n- 수원의 역사와 문화를 한눈에 볼 수 있는 박물관입니다. 가족 단위 방문객에게도 좋고, 다양한 전시와 체험 프로그램이 마련되어 있습니다.\n\n4. 광교호수공원  \n- 자연과 함께 산책하기 좋은 공원으로, 호수 주변을 걷거나 자전거를 탈 수 있습니다. 카페와 음식점도 많아 휴식하기 좋습니다.\n\n5. 행궁동 카페거리  \n- 아기자기한 카페와 맛집이 모여 있는 곳으로, 젊은 층과 관광객에게 인기 있는 장소입니다.\n\n수원 방문 시 교통편은 지하철 1호선과 분당선을 이용하면 편리하며, 버스도 잘 연결되어 있습니다. 즐거운 여행 되세요!', additional_kwargs={'refusal': None}, response_metadata={}),
 HumanMessage(content='이전에 추천한 장소 중에서 가장 인기 있는 곳은 어디인가요?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='이전에 추천해드린 장소 중에서 가장 인기 있는 곳은 **수원 화성**입니다.  \n\n수원 화성은 역사적 가치가 뛰어나고, 아름다운 성곽과 

In [16]:
# 대화 히스토리 초기화
history.clear()

In [17]:
# 대화 히스토리 출력
pprint(history.messages)

[]


# **메시지 관리 기법**

* 메시지 트리밍은 컨텍스트 윈도우의 토큰 제한을 관리하며, 시스템 메시지 포함 여부와 시작 위치 등을 세밀하게 제어할 수 있습니다.

* 장시간 진행되는 대화의 경우, 이전 대화 내용을 한 문장으로 요약하여 컨텍스트로 활용함으로써 메모리 효율성을 높일 수 있습니다.

* 대화 히스토리가 일정 길이를 초과할 경우, 요약된 내용과 최근 메시지만을 새로운 히스토리로 구성하여 컨텍스트의 품질을 유지하면서도 토큰 사용량을 최적화할 수 있습니다.

### 1. **메시지 트리밍(Message Trimming)**

* `trim_messages` 함수는 컨텍스트 윈도우의 토큰 제한을 관리하기 위한 핵심 도구로, 시스템 메시지를 포함할지 여부와 어디서부터 트리밍을 시작할지 등을 상세하게 설정할 수 있습니다.

* 트리밍 전략으로 "last" 옵션을 사용하면 가장 최근의 메시지부터 시작하여 지정된 토큰 제한에 맞춰 이전 메시지들을 선택적으로 포함시킬 수 있습니다.

In [21]:
from langchain_core.messages import trim_messages

# 메시지 목록을 정의
orginal_messages = [
    HumanMessage(content="안녕하세요. 제 이름은 홍길동입니다."),
    AIMessage(content="안녕하세요! 어떻게 도와드릴까요?"),
    HumanMessage(content="제 이름을 기억하나요?")
]

# 트리머(trimmer) 생성
# 마지막 메시지 2개만 유지하도록 설정 (token은 메시지를 나타냄) 
trimmer = trim_messages(strategy="last", max_tokens=3, token_counter=len)

trimmed_messages = trimmer.invoke(orginal_messages)

pprint(trimmed_messages)

[HumanMessage(content='안녕하세요. 제 이름은 홍길동입니다.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요! 어떻게 도와드릴까요?', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='제 이름을 기억하나요?', additional_kwargs={}, response_metadata={})]


In [19]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, trim_messages
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from pydantic import BaseModel, Field
from typing import List

# 메시지 트리밍이 적용된 인메모리 히스토리 구현
class TrimmedInMemoryHistory(BaseChatMessageHistory, BaseModel):
    messages: List[BaseMessage] = Field(default_factory=list)
    max_tokens: int = Field(default=2)  # 유지할 최대 메시지 수
    
    def __init__(self, max_tokens: int = 2, **kwargs):
        """
        TrimmedInMemoryHistory 초기화
        
        Args:
            max_tokens (int): 유지할 최대 메시지 수
            **kwargs: 추가 키워드 인자
        """
        super().__init__(max_tokens=max_tokens, **kwargs)
    
    def add_messages(self, messages: List[BaseMessage]) -> None:
        self.messages.extend(messages)
        # 메시지 추가 후 트리밍 수행
        trimmer = trim_messages(
            strategy="last",
            max_tokens=self.max_tokens,
            token_counter=len
        )
        self.messages = trimmer.invoke(self.messages)
    
    def clear(self) -> None:
        self.messages = []

# 세션 저장소
store = {}

# 세션 ID로 트리밍된 히스토리 가져오기
def get_trimmed_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = TrimmedInMemoryHistory()
    return store[session_id]

# 프롬프트 템플릿 설정
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 여행 가이드입니다. 관광객에게 유용한 정보를 제공하세요."),
    MessagesPlaceholder(variable_name="history"), 
    ("human", "{input}")
])

# 프롬프트와 llm을 연결하여 체인 생성
chain = prompt | llm

# 트리밍된 히스토리 관리 추가
chain_with_trimmed_history = RunnableWithMessageHistory(
    chain,
    get_trimmed_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# 지정된 세션 ID(tourist_1)를 사용하여 체인 실행
response = chain_with_trimmed_history.invoke(
    {
        "input": "서울에서 가볼만한 곳을 추천해주세요."
    },
    config={"configurable": {"session_id": "tourist_1"}}
)

print(f"여행 가이드 답변:\n{response.content}")

여행 가이드 답변:
서울에는 볼거리와 즐길 거리가 정말 많아요! 몇 가지 추천드릴게요.

1. 경복궁 – 조선 시대의 대표 궁궐로, 전통 건축과 궁중 문화를 체험할 수 있어요. 오전 10시와 오후 2시에 수문장 교대식도 볼 수 있습니다.

2. 북촌 한옥마을 – 전통 한옥이 잘 보존된 마을로, 골목길 산책하며 사진 찍기 좋아요. 주변에 카페와 공방도 많아요.

3. 명동 – 쇼핑과 맛집이 가득한 번화가입니다. 화장품, 패션 아이템 쇼핑하기 좋고, 길거리 음식도 다양해요.

4. N서울타워 – 남산 정상에 위치한 전망대로, 서울 전경을 한눈에 볼 수 있어요. 특히 야경이 아름답습니다.

5. 홍대 – 젊음의 거리로, 예술과 음악, 다양한 카페와 클럽이 모여 있어 활기찬 분위기를 느낄 수 있어요.

6. 동대문 디자인 플라자(DDP) – 독특한 건축물과 함께 패션, 전시, 야시장 등을 즐길 수 있는 복합 문화 공간입니다.

필요하시면 교통편이나 맛집 정보도 알려드릴게요!


In [22]:
# 대화 히스토리 출력
history = get_trimmed_session_history("tourist_1")

pprint(history.messages)

[HumanMessage(content='서울에서 가볼만한 곳을 추천해주세요.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='서울에는 볼거리와 즐길 거리가 정말 많아요! 몇 가지 추천드릴게요.\n\n1. 경복궁 – 조선 시대의 대표 궁궐로, 전통 건축과 궁중 문화를 체험할 수 있어요. 오전 10시와 오후 2시에 수문장 교대식도 볼 수 있습니다.\n\n2. 북촌 한옥마을 – 전통 한옥이 잘 보존된 마을로, 골목길 산책하며 사진 찍기 좋아요. 주변에 카페와 공방도 많아요.\n\n3. 명동 – 쇼핑과 맛집이 가득한 번화가입니다. 화장품, 패션 아이템 쇼핑하기 좋고, 길거리 음식도 다양해요.\n\n4. N서울타워 – 남산 정상에 위치한 전망대로, 서울 전경을 한눈에 볼 수 있어요. 특히 야경이 아름답습니다.\n\n5. 홍대 – 젊음의 거리로, 예술과 음악, 다양한 카페와 클럽이 모여 있어 활기찬 분위기를 느낄 수 있어요.\n\n6. 동대문 디자인 플라자(DDP) – 독특한 건축물과 함께 패션, 전시, 야시장 등을 즐길 수 있는 복합 문화 공간입니다.\n\n필요하시면 교통편이나 맛집 정보도 알려드릴게요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 294, 'prompt_tokens': 40, 'total_tokens': 334, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'syst

In [23]:
# 이전 대화 내용을 기반으로 새로운 질문을 추가하여 체인 실행
response = chain_with_trimmed_history.invoke(
    {
        "input": "이전에 추천한 장소 중에서 가장 인기 있는 곳은 어디인가요?"
    },
    config={"configurable": {"session_id": "tourist_1"}}
)

print(f"여행 가이드 답변:\n{response.content}")

여행 가이드 답변:
이전에 추천드린 장소 중에서 가장 인기 있는 곳은 경복궁과 명동, 그리고 N서울타워입니다.

- 경복궁은 한국 전통 문화와 역사를 체험할 수 있어 외국인 관광객과 현지인 모두에게 꾸준히 사랑받고 있어요.

- 명동은 쇼핑과 먹거리가 집중된 곳이라 젊은 층과 관광객들에게 매우 인기가 많고, 특히 화장품과 패션 아이템 쇼핑을 즐기기에 최적입니다.

- N서울타워는 서울의 대표적인 랜드마크로, 낮과 밤 모두 멋진 전망을 즐길 수 있어 데이트 코스나 가족 나들이 장소로도 인기가 높습니다.

이 세 곳은 서울 방문 시 꼭 들러보시는 것을 추천드려요! 필요하시면 각 장소의 방문 팁도 알려드릴게요.


In [24]:
# 대화 히스토리 출력
history = get_trimmed_session_history("tourist_1")

pprint(history.messages)

[HumanMessage(content='이전에 추천한 장소 중에서 가장 인기 있는 곳은 어디인가요?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='이전에 추천드린 장소 중에서 가장 인기 있는 곳은 경복궁과 명동, 그리고 N서울타워입니다.\n\n- 경복궁은 한국 전통 문화와 역사를 체험할 수 있어 외국인 관광객과 현지인 모두에게 꾸준히 사랑받고 있어요.\n\n- 명동은 쇼핑과 먹거리가 집중된 곳이라 젊은 층과 관광객들에게 매우 인기가 많고, 특히 화장품과 패션 아이템 쇼핑을 즐기기에 최적입니다.\n\n- N서울타워는 서울의 대표적인 랜드마크로, 낮과 밤 모두 멋진 전망을 즐길 수 있어 데이트 코스나 가족 나들이 장소로도 인기가 높습니다.\n\n이 세 곳은 서울 방문 시 꼭 들러보시는 것을 추천드려요! 필요하시면 각 장소의 방문 팁도 알려드릴게요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 191, 'prompt_tokens': 358, 'total_tokens': 549, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_a150906e27', 'id': 'chatcmpl-CEvlIIPemDEmKQpOz7aU018qOrryJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logpro

In [25]:
# 이전 대화 내용을 기반으로 새로운 질문을 추가하여 체인 실행
response = chain_with_trimmed_history.invoke(
    {
        "input": "이 장소의 정식 명칭은 무엇인가요?"
    },
    config={"configurable": {"session_id": "tourist_1"}}
)

print(f"여행 가이드 답변:\n{response.content}")

여행 가이드 답변:
말씀하신 장소가 경복궁, 명동, N서울타워 중 어느 곳인지 정확히 알려주시면 정식 명칭을 자세히 안내해드릴 수 있습니다.

- 경복궁의 정식 명칭은 '경복궁(景福宮, Gyeongbokgung Palace)'입니다.

- 명동은 지역 이름으로, 특별한 건축물 명칭이 아니라 '명동(Myeongdong)'이라고 부릅니다.

- N서울타워의 정식 명칭은 'N서울타워(N Seoul Tower)'이며, 예전에는 '남산서울타워'라고도 불렸습니다.

원하시는 장소를 알려주시면 더 구체적으로 설명해드리겠습니다!


In [26]:
# 대화 히스토리 출력
history = get_trimmed_session_history("tourist_1")

pprint(history.messages)

[HumanMessage(content='이 장소의 정식 명칭은 무엇인가요?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="말씀하신 장소가 경복궁, 명동, N서울타워 중 어느 곳인지 정확히 알려주시면 정식 명칭을 자세히 안내해드릴 수 있습니다.\n\n- 경복궁의 정식 명칭은 '경복궁(景福宮, Gyeongbokgung Palace)'입니다.\n\n- 명동은 지역 이름으로, 특별한 건축물 명칭이 아니라 '명동(Myeongdong)'이라고 부릅니다.\n\n- N서울타워의 정식 명칭은 'N서울타워(N Seoul Tower)'이며, 예전에는 '남산서울타워'라고도 불렸습니다.\n\n원하시는 장소를 알려주시면 더 구체적으로 설명해드리겠습니다!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 154, 'prompt_tokens': 256, 'total_tokens': 410, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_6d7dcc9a98', 'id': 'chatcmpl-CEvlpmDKllZT2zz2XRWSL1PRAPPXm', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--01881898-0c25-4f95-9c8c-dc8982a5c292-0', usag

### 2. **대화 요약 저장**

* 대화가 길어질 경우, 전체 대화 내용을 요약하여 컨텍스트로 활용하는 방식으로 이전 대화의 핵심을 추출합니다.

* 일반적으로 메시지 히스토리가 지정된 길이(예: 4개의 메시지)를 초과할 경우, 이전 대화들을 요약하고 가장 최근의 메시지만 유지하는 방식으로 새로운 대화 히스토리를 구성합니다.

* 이러한 요약 메모리 방식을 통해 토큰 사용량을 크게 줄이면서도 대화의 핵심 문맥을 유지할 수 있으며, 특히 장시간 진행되는 대화에서 효과적입니다.

In [27]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import List
    
class SummarizedInMemoryHistory(BaseChatMessageHistory, BaseModel):
    messages: List[BaseMessage] = Field(default_factory=list)
    summary_threshold: int = Field(default=6)  # 요약을 시작할 메시지 수
    # 요약에 사용할 LLM - lambda를 사용하여 필드가 초기화될 때마다 새로운 ChatOpenAI 인스턴스 생성
    llm: ChatOpenAI = Field(default_factory=lambda: ChatOpenAI(model="gpt-4.1-mini", temperature=0.1, top_p=0.9))  
    
    def add_messages(self, new_messages: List[BaseMessage]) -> None:
        self.messages.extend(new_messages)

        print(f"메시지 수: {len(self.messages)}")
        
        # 메시지 수가 임계값을 넘으면 요약 수행
        if len(self.messages) >= self.summary_threshold:
            # 마지막 사용자 메시지 저장 (HumanMessage, AIMessage 순서로 생성)
            last_user_message = self.messages[-2]
            last_ai_message = self.messages[-1]
            
            # 요약 생성
            summary_prompt = (
                "Distill the above chat messages into a single summary message. "
                "Include as many specific details as you can."
                "Use the original language and tone of the conversation."
            )
            
            summary_chain_messages = [
                SystemMessage(content=(
                    "You are a helpful assistant. "
                    "Your task is to summarize the conversation accurately."
                )),
                *self.messages[:-2],  # 마지막 대화 턴을 제외한 모든 메시지
                HumanMessage(content=summary_prompt)
            ]
            
            # 요약 생성
            summary = self.llm.invoke(summary_chain_messages)
            
            # 메시지 리스트 초기화 후 요약과 마지막 메시지 추가
            self.messages = [
                summary,
                last_user_message,
                last_ai_message
            ]
    
    def clear(self) -> None:
        self.messages = []

# 세션 저장소
store = {}

# 세션 ID로 요약된 히스토리 가져오기
def get_summarized_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = SummarizedInMemoryHistory()
    return store[session_id]


# 프롬프트 템플릿 설정
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 여행 가이드입니다. 관광객에게 유용한 정보를 제공하세요."),
    MessagesPlaceholder(variable_name="history"), 
    ("human", "{input}")
])

# 프롬프트와 llm을 연결하여 체인 생성
chain = prompt | llm

# 체인 구성
chain_with_summarized_history = RunnableWithMessageHistory(
    chain,
    get_summarized_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# 지정된 세션 ID(user_1)를 사용하여 체인 실행
response = chain_with_summarized_history.invoke(
    {
        "input": "서울에서 가볼만한 곳을 추천해주세요."
    },
    config={"configurable": {"session_id": "user_1"}}
)

print(f"여행 가이드 답변:\n{response.content}")

메시지 수: 2
여행 가이드 답변:
서울에서 가볼 만한 곳을 추천해드릴게요!

1. 경복궁 – 조선 시대의 대표 궁궐로, 전통 건축과 아름다운 정원을 감상할 수 있습니다. 한복을 입고 방문하면 특별한 사진도 찍을 수 있어요.

2. 북촌 한옥마을 – 전통 한옥이 잘 보존된 마을로, 골목길을 걸으며 한국의 옛 분위기를 느낄 수 있습니다.

3. 명동 – 쇼핑과 먹거리가 풍부한 번화가로, 다양한 패션 브랜드와 길거리 음식을 즐길 수 있습니다.

4. 남산서울타워 – 서울의 전경을 한눈에 볼 수 있는 전망대로, 특히 야경이 아름답습니다. 케이블카를 타고 올라가는 것도 추천해요.

5. 홍대 – 젊음의 거리로, 예술과 음악, 카페 문화가 활발한 곳입니다. 다양한 공연과 독특한 가게들을 만나볼 수 있어요.

6. 동대문 디자인 플라자(DDP) – 현대적인 건축물과 다양한 전시, 패션 마켓이 열리는 곳으로, 디자인과 문화를 체험하기 좋습니다.

서울 여행 중 궁금한 점이나 특별한 관심사가 있으면 알려주세요! 더 맞춤형으로 추천해드릴게요.


In [28]:
# 대화 히스토리 출력
history = get_summarized_session_history("user_1")

pprint(history.messages)

[HumanMessage(content='서울에서 가볼만한 곳을 추천해주세요.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='서울에서 가볼 만한 곳을 추천해드릴게요!\n\n1. 경복궁 – 조선 시대의 대표 궁궐로, 전통 건축과 아름다운 정원을 감상할 수 있습니다. 한복을 입고 방문하면 특별한 사진도 찍을 수 있어요.\n\n2. 북촌 한옥마을 – 전통 한옥이 잘 보존된 마을로, 골목길을 걸으며 한국의 옛 분위기를 느낄 수 있습니다.\n\n3. 명동 – 쇼핑과 먹거리가 풍부한 번화가로, 다양한 패션 브랜드와 길거리 음식을 즐길 수 있습니다.\n\n4. 남산서울타워 – 서울의 전경을 한눈에 볼 수 있는 전망대로, 특히 야경이 아름답습니다. 케이블카를 타고 올라가는 것도 추천해요.\n\n5. 홍대 – 젊음의 거리로, 예술과 음악, 카페 문화가 활발한 곳입니다. 다양한 공연과 독특한 가게들을 만나볼 수 있어요.\n\n6. 동대문 디자인 플라자(DDP) – 현대적인 건축물과 다양한 전시, 패션 마켓이 열리는 곳으로, 디자인과 문화를 체험하기 좋습니다.\n\n서울 여행 중 궁금한 점이나 특별한 관심사가 있으면 알려주세요! 더 맞춤형으로 추천해드릴게요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 296, 'prompt_tokens': 40, 'total_tokens': 336, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.

In [29]:
# 이전 대화 내용을 기반으로 새로운 질문을 추가하여 체인 실행
response = chain_with_summarized_history.invoke(
    {
        "input": "이전에 추천한 장소 중에서 가장 인기 있는 곳은 어디인가요?"
    },
    config={"configurable": {"session_id": "user_1"}}
)

print(f"여행 가이드 답변:\n{response.content}")

메시지 수: 4
여행 가이드 답변:
이전에 추천드린 장소 중에서 가장 인기 있는 곳은 경복궁과 명동입니다.

- **경복궁**은 한국의 대표적인 역사 문화 유적지로, 한국 전통 건축과 궁궐 문화를 체험하려는 관광객들에게 매우 인기가 많습니다. 특히 한복 체험과 함께 방문하면 더욱 특별한 경험이 됩니다.

- **명동**은 쇼핑과 먹거리의 중심지로, 국내외 관광객 모두에게 사랑받는 장소입니다. 다양한 브랜드 매장과 길거리 음식, 카페가 밀집해 있어 활기찬 분위기를 즐길 수 있습니다.

두 곳 모두 서울을 처음 방문하는 분들에게 꼭 추천드리는 명소이며, 시간과 관심사에 따라 선택하시면 좋습니다. 더 자세한 정보나 다른 인기 장소도 궁금하시면 말씀해 주세요!


In [30]:
# 대화 히스토리 출력
history = get_summarized_session_history("user_1")

pprint(history.messages)

[HumanMessage(content='서울에서 가볼만한 곳을 추천해주세요.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='서울에서 가볼 만한 곳을 추천해드릴게요!\n\n1. 경복궁 – 조선 시대의 대표 궁궐로, 전통 건축과 아름다운 정원을 감상할 수 있습니다. 한복을 입고 방문하면 특별한 사진도 찍을 수 있어요.\n\n2. 북촌 한옥마을 – 전통 한옥이 잘 보존된 마을로, 골목길을 걸으며 한국의 옛 분위기를 느낄 수 있습니다.\n\n3. 명동 – 쇼핑과 먹거리가 풍부한 번화가로, 다양한 패션 브랜드와 길거리 음식을 즐길 수 있습니다.\n\n4. 남산서울타워 – 서울의 전경을 한눈에 볼 수 있는 전망대로, 특히 야경이 아름답습니다. 케이블카를 타고 올라가는 것도 추천해요.\n\n5. 홍대 – 젊음의 거리로, 예술과 음악, 카페 문화가 활발한 곳입니다. 다양한 공연과 독특한 가게들을 만나볼 수 있어요.\n\n6. 동대문 디자인 플라자(DDP) – 현대적인 건축물과 다양한 전시, 패션 마켓이 열리는 곳으로, 디자인과 문화를 체험하기 좋습니다.\n\n서울 여행 중 궁금한 점이나 특별한 관심사가 있으면 알려주세요! 더 맞춤형으로 추천해드릴게요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 296, 'prompt_tokens': 40, 'total_tokens': 336, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.

In [31]:
# 이전 대화 내용을 기반으로 새로운 질문을 추가하여 체인 실행
response = chain_with_summarized_history.invoke(
    {
        "input": "전주에서 가볼만한 곳을 추천해주세요."
    },
    config={"configurable": {"session_id": "user_1"}}
)

print(f"여행 가이드 답변:\n{response.content}")

메시지 수: 6
여행 가이드 답변:
전주에서 가볼 만한 곳을 소개해드릴게요!

1. 전주 한옥마을 – 전통 한옥이 잘 보존된 마을로, 한국 전통 문화와 음식을 체험할 수 있는 대표 관광지입니다. 한복 체험, 전통 공예 체험, 맛있는 전주비빔밥과 한과도 즐겨보세요.

2. 경기전 – 조선 태조 이성계의 어진(초상화)을 모신 곳으로, 역사와 전통을 느낄 수 있는 중요한 문화재입니다.

3. 전동성당 – 고딕 양식의 아름다운 성당으로, 전주 한옥마을 근처에 있어 함께 방문하기 좋습니다.

4. 전주향교 – 조선 시대의 교육기관으로, 고즈넉한 분위기 속에서 한국 전통 유교 문화를 체험할 수 있습니다.

5. 남부시장 – 전주의 대표 재래시장으로, 다양한 먹거리와 쇼핑을 즐길 수 있습니다. 특히 전주 콩나물국밥, 떡갈비 등 지역 특산 음식을 맛보기에 좋습니다.

6. 덕진공원 – 호수와 정원이 아름다운 공원으로, 산책이나 휴식을 취하기에 좋은 장소입니다.

전주는 전통과 현대가 어우러진 도시로, 맛집과 문화 체험이 풍부하니 여행 계획에 참고해 보세요! 추가로 궁금한 점 있으시면 알려주세요.


In [32]:
# 대화 히스토리 출력
history = get_summarized_session_history("user_1")

pprint(history.messages)

[AIMessage(content='서울에서 가볼 만한 곳으로 경복궁, 북촌 한옥마을, 명동, 남산서울타워, 홍대, 동대문 디자인 플라자(DDP)를 추천드렸습니다. 그중 가장 인기 있는 곳은 경복궁과 명동인데요, 경복궁은 조선 시대 궁궐로 전통 건축과 한복 체험이 가능해 관광객들에게 매우 사랑받고, 명동은 쇼핑과 길거리 음식, 다양한 브랜드 매장이 밀집해 활기찬 분위기를 즐길 수 있어 국내외 방문객들에게 인기가 많습니다. 여행 중 궁금한 점이나 특별한 관심사가 있으면 언제든지 말씀해 주세요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 151, 'prompt_tokens': 576, 'total_tokens': 727, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_6d7dcc9a98', 'id': 'chatcmpl-CEw0oQS2iFQHXtETztxdPdhmH3u0a', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--f66caa65-71b5-4888-b790-02ef4c297849-0', usage_metadata={'input_tokens': 576, 'output_tokens': 151, 'total_tokens': 727, 'input_token_details': {'audio': 0, 'cache_r

In [33]:
# 이전 대화 내용을 기반으로 새로운 질문을 추가하여 체인 실행
response = chain_with_summarized_history.invoke(
    {
        "input": "두 도시에서 서로 비슷한 장소가 어디인가요?"
    },
    config={"configurable": {"session_id": "user_1"}}
)

print(f"여행 가이드 답변:\n{response.content}")

메시지 수: 5
여행 가이드 답변:
서울과 전주에서 비슷한 성격을 가진 장소를 비교해 드릴게요!

1. **전주 한옥마을 ↔ 서울 북촌 한옥마을**  
   두 곳 모두 전통 한옥이 잘 보존된 마을로, 한국 전통 건축과 문화를 체험할 수 있습니다. 한복 체험, 전통 공예, 맛집 탐방 등 비슷한 즐길 거리가 많아요.

2. **경기전 (전주) ↔ 경복궁 (서울)**  
   경기전은 조선 태조 이성계의 어진을 모신 역사적 장소이고, 경복궁은 조선 시대의 대표 궁궐입니다. 두 곳 모두 조선 왕조의 역사와 문화를 느낄 수 있는 중요한 유적지입니다.

3. **남부시장 (전주) ↔ 명동 또는 남대문시장 (서울)**  
   전주의 남부시장은 전통 재래시장으로 다양한 먹거리와 쇼핑을 즐길 수 있는 곳이고, 서울의 명동과 남대문시장도 쇼핑과 길거리 음식이 풍부한 대표적인 상업 지역입니다.

4. **덕진공원 (전주) ↔ 남산공원 (서울)**  
   덕진공원은 호수와 정원이 있는 휴식 공간이고, 남산공원은 서울의 대표적인 자연 휴식처로 산책과 전망을 즐길 수 있는 곳입니다.

이처럼 두 도시 모두 전통과 현대가 어우러진 관광 명소들이 많아 각기 다른 매력을 느끼실 수 있습니다. 여행 계획에 참고하시고, 더 궁금한 점 있으면 말씀해 주세요!


In [34]:
# 대화 히스토리 출력
history = get_summarized_session_history("user_1")

pprint(history.messages)

[AIMessage(content='서울에서 가볼 만한 곳으로 경복궁, 북촌 한옥마을, 명동, 남산서울타워, 홍대, 동대문 디자인 플라자(DDP)를 추천드렸습니다. 그중 가장 인기 있는 곳은 경복궁과 명동인데요, 경복궁은 조선 시대 궁궐로 전통 건축과 한복 체험이 가능해 관광객들에게 매우 사랑받고, 명동은 쇼핑과 길거리 음식, 다양한 브랜드 매장이 밀집해 활기찬 분위기를 즐길 수 있어 국내외 방문객들에게 인기가 많습니다. 여행 중 궁금한 점이나 특별한 관심사가 있으면 언제든지 말씀해 주세요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 151, 'prompt_tokens': 576, 'total_tokens': 727, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_6d7dcc9a98', 'id': 'chatcmpl-CEw0oQS2iFQHXtETztxdPdhmH3u0a', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--f66caa65-71b5-4888-b790-02ef4c297849-0', usage_metadata={'input_tokens': 576, 'output_tokens': 151, 'total_tokens': 727, 'input_token_details': {'audio': 0, 'cache_r

In [35]:
# 이전 대화 내용을 기반으로 새로운 질문을 추가하여 체인 실행
response = chain_with_summarized_history.invoke(
    {
        "input": "두 도시에서 각각 한 장소만 추천해주세요."
    },
    config={"configurable": {"session_id": "user_1"}}
)

print(f"여행 가이드 답변:\n{response.content}")

메시지 수: 7
여행 가이드 답변:
서울에서는 전통과 역사를 느낄 수 있는 **경복궁**을 추천드립니다. 아름다운 궁궐 건축과 함께 한복 체험도 할 수 있어 한국의 옛 문화를 생생하게 경험할 수 있습니다.

전주에서는 한국 전통 한옥의 정취를 가장 잘 느낄 수 있는 **전주 한옥마을**을 추천합니다. 고즈넉한 골목길과 전통 음식, 공예 체험까지 즐길 수 있어 전주의 매력을 한눈에 느끼실 수 있습니다.

두 곳 모두 한국의 전통 문화를 깊이 있게 체험할 수 있는 명소이니 꼭 방문해 보세요!


In [36]:
# 대화 히스토리 출력
history = get_summarized_session_history("user_1")

pprint(history.messages)

[AIMessage(content='서울과 전주에서 가볼 만한 곳을 추천드리자면, 서울에서는 경복궁(조선 시대 궁궐로 전통 건축과 한복 체험 가능), 북촌 한옥마을(전통 한옥과 문화 체험), 명동(쇼핑과 길거리 음식), 남산서울타워, 홍대, 동대문 디자인 플라자(DDP) 등이 있습니다. 전주에서는 전주 한옥마을(전통 한옥과 한복, 전통 공예 체험, 전주비빔밥 맛집), 경기전(조선 태조 이성계 어진 봉안), 전동성당(고딕 양식 성당), 전주향교(유교 문화 체험), 남부시장(재래시장과 지역 특산 음식), 덕진공원(호수와 정원 산책) 등이 대표적입니다. 두 도시에서 비슷한 장소로는 전주 한옥마을과 서울 북촌 한옥마을, 경기전과 경복궁, 남부시장과 명동 또는 남대문시장, 덕진공원과 남산공원이 각각 전통 문화, 역사 유적, 쇼핑 및 먹거리, 휴식 공간으로 대응됩니다. 각 도시마다 전통과 현대가 어우러진 매력적인 관광지가 많으니 여행 계획에 참고하시고 궁금한 점 있으면 언제든 문의해 주세요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 310, 'prompt_tokens': 944, 'total_tokens': 1254, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_a150906e27', 'id': 'chatcmpl-CEw1BTdDzwjg5RrO8sUp9GbaMLWA4', 'service_tier': 'defau

In [37]:
# 대화 히스토리 초기화
history.clear()

In [38]:
# 대화 히스토리 출력
history = get_summarized_session_history("user_1")

pprint(history.messages)

[]


---
# **[실습]**

- 메시지 트리밍과 대화 요약 저장을 결합하여 메시지를 관리하는 기능을 구현합니다. 

1. 새로운 클래스 구조

In [41]:
class TrimmedAndSummarizedHistory(BaseChatMessageHistory, BaseModel):
    """메시지 트리밍과 대화 요약을 결합한 채팅 히스토리 관리 클래스"""
    
    messages: List[BaseMessage] = Field(default_factory=list)
    max_tokens: int = Field(default=4000, description="트리밍 기준 토큰 수")
    summary_threshold: int = Field(default=10, description="요약을 위한 최소 메시지 수")
    llm: ChatOpenAI = Field(description="요약 생성용 LLM")
    summarized_messages: List[BaseMessage] = Field(default_factory=list, description="요약된 메시지들")
    
    # 토큰 카운터 (OpenAI 기본 모델 기준)
    _encoding: Optional[tiktoken.Encoding] = None
    
    class Config:
        arbitrary_types_allowed = True
    
    def __init__(self, **data):
        super().__init__(**data)
        # tiktoken 인코딩 초기화
        try:
            self._encoding = tiktoken.encoding_for_model("gpt-3.5-turbo")
        except:
            self._encoding = tiktoken.get_encoding("cl100k_base")
    
    def add_message(self, message: BaseMessage) -> None:
        """메시지 추가 및 자동 트리밍/요약 처리"""
        self.messages.append(message)
        
        # 현재 토큰 수 확인
        current_tokens = self._count_tokens(self.messages)
        
        if current_tokens > self.max_tokens:
            self._trim_and_summarize()
    
    def add_messages(self, messages: List[BaseMessage]) -> None:
        """여러 메시지 한번에 추가"""
        for message in messages:
            self.add_message(message)
    
    def get_messages(self) -> List[BaseMessage]:
        """모든 메시지 반환 (요약된 메시지 + 현재 메시지)"""
        return self.summarized_messages + self.messages
    
    def clear(self) -> None:
        """모든 메시지 삭제"""
        self.messages.clear()
        self.summarized_messages.clear()
    
    def _count_tokens(self, messages: List[BaseMessage]) -> int:
        """메시지 리스트의 총 토큰 수 계산"""
        if not self._encoding:
            # 대략적인 계산 (영어 기준 4자 = 1토큰, 한국어 기준 1.5자 = 1토큰)
            total_chars = sum(len(msg.content) for msg in messages)
            return int(total_chars / 3)  # 보수적 추정
        
        total_tokens = 0
        for message in messages:
            # 메시지 유형별 토큰 계산
            role_tokens = len(self._encoding.encode(message.__class__.__name__))
            content_tokens = len(self._encoding.encode(str(message.content)))
            total_tokens += role_tokens + content_tokens + 3  # 구조적 토큰 추가
        
        return total_tokens
    
    def _trim_and_summarize(self) -> None:
        """메시지 트리밍 및 요약 처리"""
        if len(self.messages) < self.summary_threshold:
            # 메시지 수가 적으면 단순 트리밍
            self._simple_trim()
            return
        
        # 트리밍할 메시지 식별
        messages_to_remove = self._identify_messages_to_remove()
        
        if not messages_to_remove:
            return
        
        # 제거될 메시지들 요약
        summary = self._summarize_messages(messages_to_remove)
        
        if summary:
            # 요약을 시스템 메시지로 저장
            summary_message = SystemMessage(
                content=f"[대화 요약] {summary}",
                additional_kwargs={"summary_timestamp": self._get_timestamp()}
            )
            self.summarized_messages.append(summary_message)
        
        # 메시지 트리밍 실행
        self._remove_messages(messages_to_remove)
    
    def _identify_messages_to_remove(self) -> List[BaseMessage]:
        """제거할 메시지들 식별"""
        current_tokens = self._count_tokens(self.messages)
        target_tokens = int(self.max_tokens * 0.7)  # 70% 수준으로 트리밍
        
        messages_to_remove = []
        tokens_to_remove = current_tokens - target_tokens
        
        # 오래된 메시지부터 제거 (최근 메시지는 보존)
        for i, message in enumerate(self.messages):
            if tokens_to_remove <= 0:
                break
            
            # 시스템 메시지는 보존
            if isinstance(message, SystemMessage):
                continue
            
            message_tokens = self._count_tokens([message])
            messages_to_remove.append(message)
            tokens_to_remove -= message_tokens
        
        return messages_to_remove
    
    def _summarize_messages(self, messages: List[BaseMessage]) -> Optional[str]:
        """메시지들을 요약"""
        if not messages:
            return None
        
        # 요약 프롬프트 템플릿
        summary_prompt = ChatPromptTemplate.from_messages([
            ("system", """다음 대화 내용을 간결하고 핵심적인 정보만 포함하여 요약해주세요.
            
요약 지침:
- 주요 토픽과 결론만 포함
- 구체적인 정보나 데이터가 있다면 보존
- 대화의 맥락과 흐름 유지
- 불필요한 인사말이나 반복적인 내용은 제외
- 한국어로 자연스럽게 요약"""),
            ("user", "요약할 대화:\n{conversation}")
        ])
        
        # 대화 내용 구성
        conversation_text = ""
        for msg in messages:
            role = self._get_role_name(msg)
            conversation_text += f"{role}: {msg.content}\n"
        
        try:
            # 요약 생성
            chain = summary_prompt | self.llm
            response = chain.invoke({"conversation": conversation_text})
            return response.content.strip()
        
        except Exception as e:
            print(f"요약 생성 중 오류 발생: {e}")
            return f"대화 요약 (메시지 {len(messages)}개 포함)"
    
    def _get_role_name(self, message: BaseMessage) -> str:
        """메시지 타입별 역할 이름 반환"""
        if isinstance(message, HumanMessage):
            return "사용자"
        elif isinstance(message, AIMessage):
            return "AI"
        elif isinstance(message, SystemMessage):
            return "시스템"
        else:
            return "기타"
    
    def _simple_trim(self) -> None:
        """단순 트리밍 (요약 없이)"""
        target_tokens = int(self.max_tokens * 0.7)
        
        while self._count_tokens(self.messages) > target_tokens and self.messages:
            # 가장 오래된 메시지부터 제거 (시스템 메시지 제외)
            for i, message in enumerate(self.messages):
                if not isinstance(message, SystemMessage):
                    self.messages.pop(i)
                    break
    
    def _remove_messages(self, messages_to_remove: List[BaseMessage]) -> None:
        """지정된 메시지들 제거"""
        for message in messages_to_remove:
            if message in self.messages:
                self.messages.remove(message)
    
    def _get_timestamp(self) -> str:
        """현재 타임스탬프 반환"""
        from datetime import datetime
        return datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    def get_statistics(self) -> dict:
        """히스토리 통계 정보 반환"""
        return {
            "current_messages": len(self.messages),
            "summarized_messages": len(self.summarized_messages),
            "total_messages": len(self.get_messages()),
            "current_tokens": self._count_tokens(self.messages),
            "max_tokens": self.max_tokens,
            "token_usage_ratio": self._count_tokens(self.messages) / self.max_tokens
        }
    
    def print_summary(self) -> None:
        """히스토리 요약 정보 출력"""
        stats = self.get_statistics()
        print("=== 채팅 히스토리 현황 ===")
        print(f"현재 메시지 수: {stats['current_messages']}")
        print(f"요약된 메시지 수: {stats['summarized_messages']}")
        print(f"총 메시지 수: {stats['total_messages']}")
        print(f"현재 토큰 사용량: {stats['current_tokens']} / {stats['max_tokens']}")
        print(f"토큰 사용률: {stats['token_usage_ratio']:.1%}")
        
        if self.summarized_messages:
            print("\n=== 저장된 요약 ===")
            for i, msg in enumerate(self.summarized_messages, 1):
                print(f"{i}. {msg.content[:100]}...")


# 사용 예시
if __name__ == "__main__":
    from langchain_openai import ChatOpenAI
    
    # LLM 초기화
    llm = ChatOpenAI(
        model="gpt-3.5-turbo",
        temperature=0.3,
        # api_key="your-openai-api-key"  # 실제 사용시 API 키 설정
    )
    
    # 히스토리 관리자 초기화
    history = TrimmedAndSummarizedHistory(
        max_tokens=1000,      # 작은 값으로 테스트
        summary_threshold=4,   # 4개 메시지부터 요약
        llm=llm
    )
    
    # 테스트 메시지들 추가
    test_messages = [
        HumanMessage(content="안녕하세요! 파이썬 프로그래밍에 대해 질문이 있어요."),
        AIMessage(content="안녕하세요! 파이썬 관련 질문을 언제든 해주세요. 도와드리겠습니다."),
        HumanMessage(content="리스트 컴프리헨션에 대해 설명해주실 수 있나요?"),
        AIMessage(content="네! 리스트 컴프리헨션은 기존 리스트를 기반으로 새로운 리스트를 간결하게 생성하는 방법입니다. 예를 들어 [x*2 for x in range(5)]는 [0, 2, 4, 6, 8]을 만듭니다."),
        HumanMessage(content="조건문도 사용할 수 있나요?"),
        AIMessage(content="네, 조건문도 사용할 수 있습니다. [x for x in range(10) if x % 2 == 0]처럼 if 조건을 추가할 수 있어요."),
        HumanMessage(content="딕셔너리 컴프리헨션도 있나요?"),
        AIMessage(content="네! 딕셔너리 컴프리헨션도 있습니다. {x: x**2 for x in range(5)}처럼 사용할 수 있어요."),
    ]
    
    # 메시지 추가 및 자동 트리밍/요약 테스트
    for msg in test_messages:
        history.add_message(msg)
        print(f"메시지 추가 후 토큰 수: {history._count_tokens(history.messages)}")
    
    # 결과 확인
    history.print_summary()
    
    print("\n=== 전체 메시지 (요약 포함) ===")
    for i, msg in enumerate(history.get_messages(), 1):
        role = history._get_role_name(msg)
        content = msg.content[:100] + "..." if len(msg.content) > 100 else msg.content
        print(f"{i}. {role}: {content}")


메시지 추가 후 토큰 수: 32
메시지 추가 후 토큰 수: 71
메시지 추가 후 토큰 수: 101
메시지 추가 후 토큰 수: 186
메시지 추가 후 토큰 수: 203
메시지 추가 후 토큰 수: 256
메시지 추가 후 토큰 수: 282
메시지 추가 후 토큰 수: 334
=== 채팅 히스토리 현황 ===
현재 메시지 수: 8
요약된 메시지 수: 0
총 메시지 수: 8
현재 토큰 사용량: 334 / 1000
토큰 사용률: 33.4%

=== 전체 메시지 (요약 포함) ===
1. 사용자: 안녕하세요! 파이썬 프로그래밍에 대해 질문이 있어요.
2. AI: 안녕하세요! 파이썬 관련 질문을 언제든 해주세요. 도와드리겠습니다.
3. 사용자: 리스트 컴프리헨션에 대해 설명해주실 수 있나요?
4. AI: 네! 리스트 컴프리헨션은 기존 리스트를 기반으로 새로운 리스트를 간결하게 생성하는 방법입니다. 예를 들어 [x*2 for x in range(5)]는 [0, 2, 4, 6, 8]을...
5. 사용자: 조건문도 사용할 수 있나요?
6. AI: 네, 조건문도 사용할 수 있습니다. [x for x in range(10) if x % 2 == 0]처럼 if 조건을 추가할 수 있어요.
7. 사용자: 딕셔너리 컴프리헨션도 있나요?
8. AI: 네! 딕셔너리 컴프리헨션도 있습니다. {x: x**2 for x in range(5)}처럼 사용할 수 있어요.


2. 메시지 처리 로직
- 새 메시지 추가시 max_tokens 체크
- 트리밍 발생하면 제거될 메시지 식별
- 제거 예정 메시지들은 요약하여 summarized_messages에 저장
- 현재 메시지는 트리밍된 상태로 유지

3. 요약 프로세스
- 트리밍으로 제거될 메시지들만 선별
- 선별된 메시지들에 대해 summary_chain 실행
- 요약본을 시스템 메시지로 변환하여 저장

In [ ]:
# 여기에 코드를 작성하세요.
